# VM Resource Planner (Reactive, no forecast)

Notebook này chạy pipeline `vm_resource_planner.py` ở chế độ **reactive**: không dùng forecast, coi `y_test` như luồng đo đạc đến từng timestamp. LP được giải **mỗi timestamp** (không bucket time), áp dụng chi phí thuê VM và chi phí switching trong `VMs_type.json`.

Các bước chính:
1. Load cấu hình & VM catalog (có `cost_per_hour`, `switching_cost`).
2. Stream `y_test` thành chuỗi đo đạc thời gian thực (không nhìn trước tương lai).
3. Chuyển đo đạc → nhu cầu overflow CPU/RAM so với host.
4. Giải LP mỗi timestamp, ghi lại allocation, switching cost, utilization, SLA.
5. Lưu schedule per-step và tổng hợp metrics cho toàn bộ tập test.



In [1]:
import json
import importlib
from pathlib import Path

import pandas as pd

# Reload module to ensure latest changes are loaded
import vm_resource_planner
importlib.reload(vm_resource_planner)

from vm_resource_planner import (
    load_vm_catalog,
    load_ground_truth_df,
    convert_forecasts_to_requirements,
    build_reactive_schedule,
    compute_lp_metrics,
    HOST_SPEC,
    VM_TYPES_FILE,
    RESULTS_DIR,
)

RESULTS_DIR.mkdir(exist_ok=True)
print("✓ Libraries & planner helpers loaded (reactive mode)")


✓ Libraries & planner helpers loaded (reactive mode)


In [2]:
vm_catalog = load_vm_catalog(VM_TYPES_FILE)

print("VM catalog (with switching_cost):")
for spec in vm_catalog:
    print(
        f"  • {spec['name']}: {spec['vcpus']} vCPUs, {spec['memory_gb']} GB, "
        f"${spec['cost_per_hour']}/h, switching_cost={spec.get('switching_cost',0)}"
    )

HOST_SPEC


VM catalog (with switching_cost):
  • B2s: 2 vCPUs, 4 GB, $0.0416/h, switching_cost=0
  • D2s_v3: 2 vCPUs, 8 GB, $0.096/h, switching_cost=0
  • D8s_v3: 8 vCPUs, 64 GB, $0.384/h, switching_cost=0
  • D32s_v3: 32 vCPUs, 128 GB, $1.536/h, switching_cost=0


{'total_cpu_cores': 1,
 'total_memory_gb': 4,
 'cpu_threshold_pct': 70,
 'memory_threshold_pct': 75}

In [3]:
ground_truth_df = load_ground_truth_df()
ground_truth_df.head()


,timestamp,memory_usage_pct,cpu_total_usage,system_load
0,2024-01-01 00:00:00,0.027578,-0.232042,0.527563
1,2024-01-01 00:00:30,0.027839,-0.485797,0.431887
2,2024-01-01 00:01:00,0.024400,-0.470571,0.001342
3,2024-01-01 00:01:30,0.019491,-0.501022,-0.237850
4,2024-01-01 00:02:00,0.031230,-0.419821,-0.142173


In [4]:
# Chuyển ground-truth đo đạc thành nhu cầu overflow
requirements_df = convert_forecasts_to_requirements(ground_truth_df, HOST_SPEC)
requirements_df[[
    'timestamp',
    'cpu_total_usage', 'cpu_required_cores', 'cpu_overflow_cores',
    'memory_usage_pct', 'memory_required_gb', 'memory_overflow_gb'
]].head()


,timestamp,cpu_total_usage,cpu_required_cores,cpu_overflow_cores,memory_usage_pct,memory_required_gb,memory_overflow_gb
0,2024-01-01 00:00:00,-0.232042,0.527563,0.0,0.027578,0.001103,0.0
1,2024-01-01 00:00:30,-0.485797,0.431887,0.0,0.027839,0.001114,0.0
2,2024-01-01 00:01:00,-0.470571,0.001342,0.0,0.024400,0.000976,0.0
3,2024-01-01 00:01:30,-0.501022,0.000000,0.0,0.019491,0.000780,0.0
4,2024-01-01 00:02:00,-0.419821,0.000000,0.0,0.031230,0.001249,0.0


In [5]:
schedule_df = build_reactive_schedule(requirements_df, vm_catalog)
schedule_df.head()


,timestamp,cpu_overflow_cores,memory_overflow_gb,min_overload_plan,min_overload_cost_per_hour,min_cost_plan,min_cost_cost_per_hour,vm_plan,vm_total_count,vm_cost_per_hour,vm_cost_per_step,switching_cost,total_cost_per_hour,total_cost_per_step,vm_cpu_allocated,vm_mem_allocated,sla_violation,cpu_utilization_pct,mem_utilization_pct
0,2024-01-01 00:00:00,0.0,0.0,Host only,0.0,Host only,0.0,Host only,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0
1,2024-01-01 00:00:30,0.0,0.0,Host only,0.0,Host only,0.0,Host only,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0
2,2024-01-01 00:01:00,0.0,0.0,Host only,0.0,Host only,0.0,Host only,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0
3,2024-01-01 00:01:30,0.0,0.0,Host only,0.0,Host only,0.0,Host only,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0
4,2024-01-01 00:02:00,0.0,0.0,Host only,0.0,Host only,0.0,Host only,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0


In [6]:
metrics = compute_lp_metrics(schedule_df)
metrics


{'total_vm_cost': 167.02399999999997,
 'total_switching_cost': 0.0,
 'total_cost': 167.02399999999997,
 'total_cost_step': 1.3918666666666666,
 'total_vms_sum': 4015,
 'avg_vms': 0.23411078717201167,
 'mean_cpu_utilization_pct': 8.56099155045171,
 'mean_mem_utilization_pct': 0.0,
 'sla_violations': 0,
 'sla_violation_rate': 0.0}

In [7]:
# Lưu kết quả reactive
report_path = RESULTS_DIR / "vm_resource_planning_notebook_reactive.json"
schedule_path = RESULTS_DIR / "vm_schedule_notebook_reactive.csv"

schedule_records = schedule_df.copy()
schedule_records['timestamp'] = pd.to_datetime(schedule_records['timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S')

report_payload = {
    'model': 'vm_resource_planner_notebook_reactive',
    'host_spec': HOST_SPEC,
    'metrics': metrics,
    'schedule': schedule_records.to_dict(orient='records'),
}

with open(report_path, 'w') as f:
    json.dump(report_payload, f, indent=2)

schedule_df.to_csv(schedule_path, index=False)

report_path, schedule_path


(WindowsPath('E:/PROJECTS/Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure/forecast_result/vm_resource_planning_notebook_reactive.json'),
 WindowsPath('E:/PROJECTS/Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure/forecast_result/vm_schedule_notebook_reactive.csv'))

In [8]:
# Hoàn tất
print("Reactive LP schedule + metrics saved.")
print("Report:", report_path)
print("CSV:", schedule_path)


Reactive LP schedule + metrics saved.
Report: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\vm_resource_planning_notebook_reactive.json
CSV: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\vm_schedule_notebook_reactive.csv
